In [1]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

tables = spark.sql(f'SHOW TABLES IN {LAB}').collect()

for table_row in tables:
    spark.sql(f'DROP TABLE IF EXISTS {LAB}.{table_row.tableName}')
    print(f"dropped {table_row}")


spark.sql(f'DROP VOLUME IF EXISTS {LAB}.files')
spark.sql(f'DROP SCHEMA IF EXISTS {LAB}')
print(f"{LAB} is now clear, ready for recreation")

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'

print(f"{LAB} is created")


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

dbutils.widgets.dropdown("Is cluster mode?", "False", ["True","False"])
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")

car_workshop.lab is now clear, ready for recreation
car_workshop.lab is created
lab schema: car_workshop.lab, lab volume: /Volumes/car_workshop/lab/files


/Users/mtwa/Desktop/git/fake_car_workshop/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [2]:
# build an artificially skewed table for the exercises: 80% of rows -> one hot product
items = spark.table(f'{CATALOG}.fact.fact_sales_items')
hot_product = items.groupBy('product_id').count().orderBy(F.desc('count')).first()['product_id']

(items
 .withColumn('product_id',
             F.when(F.rand(seed=42) < 0.8, F.lit(hot_product)).otherwise(F.col('product_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.skewed_sales_items'))

skewed = spark.table(f'{LAB}.skewed_sales_items')
display(skewed.groupBy('product_id').agg(F.count('*').alias('cnt')).orderBy(F.desc('cnt')).limit(5))

,product_id,cnt
0,355,15905905
1,342,8769
2,359,8674
3,191,8640
4,385,8618


In [3]:
fact_schema = f"{CATALOG}.fact"
tables = spark.sql(f'SHOW TABLES IN {fact_schema}').collect()

for table_row in tables:
    table = spark.sql(f'DESCRIBE TABLE {fact_schema}.{table_row.tableName}')
    print(f"table: {table_row}")
    display(table)

table: Row(database='fact', tableName='fact_appointments', isTemporary=False)


,col_name,data_type,comment
0,appointment_id,bigint,None
1,customer_id,bigint,None
2,vehicle_id,bigint,None
3,location_id,bigint,None
4,service_id,bigint,None
5,booking_date,date,None
6,appointment_date,date,None
7,status,string,None
8,booking_channel,string,None
9,notes,string,None


table: Row(database='fact', tableName='fact_customer_feedback', isTemporary=False)


,col_name,data_type,comment
0,feedback_id,bigint,None
1,customer_id,bigint,None
2,location_id,bigint,None
3,work_order_id,bigint,None
4,feedback_date,date,None
5,rating,bigint,None
6,comment,string,None
7,category,string,None
8,channel,string,None
9,_rescued_data,string,None


table: Row(database='fact', tableName='fact_employee_schedules', isTemporary=False)


...

,col_name,data_type,comment
0,transaction_id,bigint,None
1,transaction_code,string,None
2,location_id,bigint,None
3,customer_id,bigint,None
4,employee_id,bigint,None
5,transaction_date,date,None
6,payment_method,string,None
7,receipt_number,string,None
8,year,int,None
9,month,int,None


table: Row(database='fact', tableName='fact_work_order_items', isTemporary=False)


,col_name,data_type,comment
0,wo_item_id,bigint,None
1,work_order_id,bigint,None
2,item_type,string,None
3,service_id,bigint,None
4,product_id,bigint,None
5,quantity,bigint,None
6,unit_price_net,double,None
7,value_net,double,None
8,vat_rate,bigint,None
9,value_gross,double,None


table: Row(database='fact', tableName='fact_work_orders', isTemporary=False)


,col_name,data_type,comment
0,work_order_id,bigint,None
1,work_order_code,string,None
2,location_id,bigint,None
3,customer_id,bigint,None
4,vehicle_id,bigint,None
5,mechanic_id,bigint,None
6,reception_date,date,None
7,completion_date,date,None
8,status,string,None
9,mileage_at_reception,bigint,None


In [4]:
%sql
select distinct sale_date from car_workshop.fact.fact_invoices 
order by sale_date asc

,sale_date
0,2024-12-30
1,2024-12-31
2,2025-01-01
3,2025-01-02
4,2025-01-03
5,2025-01-04
6,2025-01-05
7,2025-01-06
8,2025-01-07
9,2025-01-08


In [5]:
%sql
create table car_workshop.lab.fact_invoices as
with cte as (
  select 
    *,
    month(sale_date) as sales_month,
    case when month(sale_date) between 1 and 8  then 'skew_1'
    when month(sale_date) between 8 and 9 then 'skew_2'
    when month(sale_date) between 10 and 11 then 'skew_3'
    when month(sale_date) between 12 and 12 then 'skew_4'
    end as skew_test
  from car_workshop.fact.fact_invoices
),
skew_1_multiplied as (
  select cte.* 
  from cte
  CROSS JOIN LATERAL explode(sequence(1, 150)) AS t(multiplier)
  where skew_test = 'skew_1'
),
other_skews as (
  select *
  from cte
  where skew_test != 'skew_1'
),
combined as (
  select * from skew_1_multiplied
  union all
  select * from other_skews
)
-- check the skew
-- select skew_test, count(skew_test) as row_count 
-- from combined
-- group by skew_test
select * from combined

,num_affected_rows,num_inserted_rows


In [6]:
%sql
select skew_test, count(*) from car_workshop.lab.fact_invoices
group by skew_test

,skew_test,count(*)
0,skew_1,642003750
1,skew_3,610173
2,skew_4,329863
3,skew_2,299939


In [7]:
%sql
select count(*) from car_workshop.lab.fact_invoices

,count(*)
0,643243725


In [8]:
from pyspark.sql.window import Window

inv = spark.table(f'{LAB}.fact_invoices')

# pathological: ALL 643243725 rows of skew_1 must land in ONE task and get sorted there
w_skew = Window.partitionBy('skew_test').orderBy('sale_date')
timed('window over skew_test (4 tasks, one with 643243725 rows)',
      lambda: inv.withColumn('rn', F.row_number().over(w_skew))
                 .agg(F.max('rn')).collect())

# same work, healthy key: invoice_id has high cardinality -> spreads evenly
w_ok = Window.partitionBy('invoice_id').orderBy('sale_date')
timed('window over invoice_id (well distributed)',
      lambda: inv.withColumn('rn', F.row_number().over(w_ok))
                 .agg(F.max('rn')).collect())

window over skew_test (4 tasks, one with 643243725 rows): 244.5s
window over invoice_id (well distributed): 26.0s


[Row(max(rn)=150)]

In [9]:
%sql

select * from car_workshop.fact.fact_invoices limit 1000

,invoice_id,invoice_code,document_type,source_type,source_id,customer_id,location_id,issue_date,sale_date,payment_due_date,value_net,value_vat,value_gross,status,year,month,_rescued_data
0,22010000001,INV/2026/22010000001,vat_invoice,sales,22010004286,431232,13,2026-01-10,2026-01-10,2026-01-24,40.78,9.38,50.16,paid,2026,1,None
1,22010000002,INV/2026/22010000002,receipt,sales,22010007799,185731,53,2026-01-10,2026-01-08,2026-01-10,17.43,4.01,21.44,paid,2026,1,None
2,22010000003,INV/2026/22010000003,vat_invoice,sales,22010005110,47169,16,2026-01-10,2026-01-08,2026-01-10,307.29,70.68,377.97,paid,2026,1,None
3,22010000004,INV/2026/22010000004,receipt,sales,22010003329,302718,96,2026-01-10,2026-01-08,2026-01-10,147.74,33.98,181.72,paid,2026,1,None
4,22010000005,INV/2026/22010000005,vat_invoice,sales,22010003114,483835,84,2026-01-10,2026-01-08,2026-01-24,17.98,4.14,22.12,paid,2026,1,None
5,22010000006,INV/2026/22010000006,receipt,sales,22010007989,159910,50,2026-01-10,2026-01-10,2026-02-09,167.70,38.57,206.27,paid,2026,1,None
6,22010000007,INV/2026/22010000007,receipt,sales,22010010123,178321,82,2026-01-10,2026-01-10,2026-01-10,49.52,11.39,60.91,cancelled,2026,1,None
7,22010000008,INV/2026/22010000008,receipt,sales,22010005377,153179,9,2026-01-10,2026-01-08,2026-01-10,29.28,6.73,36.01,paid,2026,1,None
8,22010000009,INV/2026/22010000009,receipt,sales,22010002618,99936,72,2026-01-10,2026-01-10,2026-01-17,66.60,15.32,81.92,paid,2026,1,None
9,22010000010,INV/2026/22010000010,receipt,sales,22010007922,366988,88,2026-01-10,2026-01-09,2026-01-10,476.55,109.61,586.16,paid,2026,1,None


In [10]:
%sql

select * from car_workshop.fact.fact_sales_transactions limit 1000

,transaction_id,transaction_code,location_id,customer_id,employee_id,transaction_date,payment_method,receipt_number,year,month,_rescued_data
0,18400000001,TRX-18400000001,42,NaN,475,2025-01-14,cash,REC/042/18400000001,2025,1,None
1,18400000002,TRX-18400000002,14,48098.0,160,2025-01-14,card,REC/014/18400000002,2025,1,None
2,18400000003,TRX-18400000003,53,NaN,607,2025-01-14,card,REC/053/18400000003,2025,1,None
3,18400000004,TRX-18400000004,48,NaN,545,2025-01-14,bank_transfer,REC/048/18400000004,2025,1,None
4,18400000005,TRX-18400000005,30,NaN,334,2025-01-14,card,REC/030/18400000005,2025,1,None
5,18400000006,TRX-18400000006,83,NaN,986,2025-01-14,card,REC/083/18400000006,2025,1,None
6,18400000007,TRX-18400000007,84,NaN,1000,2025-01-14,card,REC/084/18400000007,2025,1,None
7,18400000008,TRX-18400000008,22,404088.0,259,2025-01-14,BLIK,REC/022/18400000008,2025,1,None
8,18400000009,TRX-18400000009,93,281209.0,1092,2025-01-14,card,REC/093/18400000009,2025,1,None
9,18400000010,TRX-18400000010,55,382384.0,635,2025-01-14,cash,REC/055/18400000010,2025,1,None


In [11]:
# skew a REAL FK: 80% of invoices point to one hot transaction
trx = spark.table(f'{CATALOG}.fact.fact_sales_transactions')

# invoices have NO transaction_id - the FK is polymorphic (source_type + source_id),
# so keep only the 'sales' branch and give the key its real name
inv_base = (spark.table(f'{CATALOG}.fact.fact_invoices')
            .filter("source_type = 'sales'")                      # ~85% of invoices
            .withColumnRenamed('source_id', 'transaction_id'))

# any existing id will do as the hot key - 80% of rows get overwritten anyway
hot_trx = inv_base.select('transaction_id').first()['transaction_id']

(inv_base
 .withColumn('transaction_id',
             F.when(F.rand(seed=42) < 0.8, F.lit(hot_trx)).otherwise(F.col('transaction_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.invoices_hot_fk'))

# both sides are large facts -> no broadcast escape, real shuffle join
timed('fact-to-fact join on skewed FK',
      lambda: spark.table(f'{LAB}.invoices_hot_fk')
                   .join(trx, 'transaction_id')
                   .agg(F.sum('value_gross')).collect())

fact-to-fact join on skewed FK: 2.7s


[Row(sum(value_gross)=856442817.4698143)]

In [12]:
%sql

select * from car_workshop.lab.invoices_hot_fk limit 1000

,invoice_id,invoice_code,document_type,source_type,transaction_id,customer_id,location_id,issue_date,sale_date,payment_due_date,value_net,value_vat,value_gross,status,year,month,_rescued_data
0,18440000001,INV/2025/18440000001,receipt,sales,21740003240,5208,66,2025-01-18,2025-01-17,2025-02-17,18.68,4.30,22.98,pending,2025,1,None
1,18440000002,INV/2025/18440000002,receipt,sales,21740003240,71467,75,2025-01-18,2025-01-16,2025-01-18,428.08,98.46,526.54,paid,2025,1,None
2,18440000003,INV/2025/18440000003,receipt,sales,21740003240,38252,97,2025-01-18,2025-01-17,2025-01-25,10.00,2.30,12.30,paid,2025,1,None
3,18440000004,INV/2025/18440000004,receipt,sales,21740003240,520,16,2025-01-18,2025-01-17,2025-01-18,84.69,19.48,104.17,pending,2025,1,None
4,18440000006,INV/2025/18440000006,vat_invoice,sales,21740003240,499980,92,2025-01-18,2025-01-17,2025-01-18,325.42,74.85,400.27,paid,2025,1,None
5,18440000007,INV/2025/18440000007,receipt,sales,21740003240,438163,77,2025-01-18,2025-01-18,2025-01-18,120.21,27.65,147.86,overdue,2025,1,None
6,18440000009,INV/2025/18440000009,vat_invoice,sales,21740003240,318193,6,2025-01-18,2025-01-16,2025-01-18,51.49,11.84,63.33,paid,2025,1,None
7,18440000011,INV/2025/18440000011,vat_invoice,sales,21740003240,213704,63,2025-01-18,2025-01-17,2025-01-18,261.92,60.24,322.16,paid,2025,1,None
8,18440000012,INV/2025/18440000012,receipt,sales,18440000328,407504,63,2025-01-18,2025-01-18,2025-01-18,55.46,12.76,68.22,paid,2025,1,None
9,18440000013,INV/2025/18440000013,receipt,sales,21740003240,295123,51,2025-01-18,2025-01-17,2025-01-18,413.59,95.13,508.72,cancelled,2025,1,None


In [13]:
inv = spark.table(f'{LAB}.fact_invoices')

# stage 1: pre-aggregate per (key, salt) -> stage 2: final agg per key
salted = (inv
    .withColumn('salt', (F.rand(seed=7) * 32).cast('int'))
    .groupBy('skew_test', 'salt').agg(F.sum('value_gross').alias('partial'))
    .groupBy('skew_test').agg(F.sum('partial').alias('total')))

# correctness check - salting must NOT change the numbers
plain = inv.groupBy('skew_test').agg(F.sum('value_gross').alias('total'))
display(salted.orderBy('skew_test'))
display(plain.orderBy('skew_test'))


,skew_test,total
0,skew_1,1.172199e+11
1,skew_2,5.460333e+07
2,skew_3,1.112896e+08
3,skew_4,6.022541e+07


,skew_test,total
0,skew_1,1.172199e+11
1,skew_2,5.460333e+07
2,skew_3,1.112896e+08
3,skew_4,6.022541e+07


In [14]:

# SERVERLESS variant - salting is plain DataFrame code, so exercise CORRECTNESS:
# the salted join must return exactly the same aggregate as the plain join
# (this is the part people get wrong - forgetting the dim replication).
# The performance effect needs a forced SMJ -> classic cluster.
SALT = 16 # why 16 ?
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')

fact_salted_sl = (skewed_df
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_salted_sl = (products_df
    .crossJoin(spark.range(SALT).withColumnRenamed('id', 'salt'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain = skewed_df.join(products_df, 'product_id').agg(F.round(F.sum('value_net'), 2)).first()[0]
salted = fact_salted_sl.join(dim_salted_sl, 'k_salt').agg(F.round(F.sum('value_net'), 2)).first()[0]
print(f'plain join:  {plain:,}')
print(f'salted join: {salted:,}  -> identical: {plain == salted}')

plain join:  11,235,536,805.32
salted join: 11,235,536,805.32  -> identical: True


In [15]:
fact_salted_sl.display()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `display` is not supported.

In [16]:
dim_salted_sl.display()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `display` is not supported.

In [17]:
# two-stage (salted) AGGREGATION - for distributive aggs: sum/count/min/max
# stage 1: partial agg on (key, salt) -> stage 2: final agg on key
partial = (skewed
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .groupBy('product_id', 'salt')
    .agg(F.sum('value_net').alias('partial_sum'), F.count('*').alias('partial_cnt')))

final = (partial.groupBy('product_id')
    .agg(F.sum('partial_sum').alias('revenue'),
         (F.sum('partial_sum') / F.sum('partial_cnt')).alias('avg_value')))  # avg = sum/count!

display(final.orderBy(F.desc('revenue')).limit(5))
# note: count(distinct) can NOT be salted this way

,product_id,revenue,avg_value
0,355,8.993106e+09,565.394179
1,342,5.017792e+06,572.219368
2,21,4.973120e+06,578.808145
3,156,4.941847e+06,575.436265
4,359,4.919088e+06,567.107271


In [ ]:
# SERVERLESS variant - hybrid salting without the conf toggles: verify the logic
# (row counts must match the plain join), benchmark later on the classic cluster.
SALT = 16 # to be changed into automatically computed value based on the skewness of the fact table

skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')
salt_range_sl = spark.range(SALT).withColumnRenamed('id', 'salt')

counts_sl = skewed_df.groupBy('product_id').agg(F.count('*').alias('cnt'))
median_sl = counts_sl.select(F.expr('percentile_approx(cnt, 0.5)')).first()[0]
hot_sl = counts_sl.filter(F.col('cnt') > 20 * median_sl).select('product_id')

fact2_sl = (skewed_df
    .join(F.broadcast(hot_sl.withColumn('is_hot', F.lit(True))), 'product_id', 'left')
    .withColumn('salt', F.when(F.col('is_hot').isNotNull(),
                               (F.rand(seed=7) * SALT).cast('int')).otherwise(F.lit(0)))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_hot_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'inner').crossJoin(salt_range_sl)
dim_cold_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'left_anti').withColumn('salt', F.lit(0))
dim2_sl = (dim_hot_sl.unionByName(dim_cold_sl)
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain_cnt = skewed_df.join(products_df, 'product_id').count()
hybrid_cnt = fact2_sl.join(dim2_sl, 'k_salt').count()
print(f'plain: {plain_cnt:,} rows, hybrid-salted: {hybrid_cnt:,} -> identical: {plain_cnt == hybrid_cnt}')

plain: 19,871,950 rows, hybrid-salted: 19,871,950 -> identical: True
